# Visualizing Numerical Data

This notebook explores different techniques for visualizing numerical data using popular Python visualization libraries. We'll cover various plot types and best practices to effectively represent and analyze numerical data.

## 1. Import Required Libraries

In [ ]:
# Import core visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

# Set matplotlib style and configure visualizations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style="whitegrid", palette="muted")

# Increase default figure size for better readability
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

# Import sample datasets
from seaborn import load_dataset

# Display plots inline
%matplotlib inline

## 2. Understanding Numerical Data Types

Numerical data comes in different types, each requiring appropriate visualization techniques:

1. **Continuous data**: Can take any value within a range (e.g., height, weight, temperature)
2. **Discrete data**: Takes on distinct, separate values (e.g., counts, integers)
3. **Interval data**: Values where the difference between points is meaningful, but there's no true zero point (e.g., temperature in Celsius, dates)
4. **Ratio data**: Has a meaningful zero and where ratios between values are meaningful (e.g., height, weight, age)

Let's create some synthetic datasets to demonstrate these different types of numerical data.

In [ ]:
# Set a random seed for reproducibility
np.random.seed(42)

# Create sample datasets for different numerical data types
n_samples = 1000

# Continuous data (heights in cm)
heights = np.random.normal(170, 10, n_samples)

# Discrete data (number of children in families)
children = np.random.poisson(1.8, n_samples)

# Temperature data (interval data - Celsius)
temperatures = np.random.normal(15, 8, n_samples)

# Income data (ratio data - with meaningful zero and ratios)
income = np.random.lognormal(10, 0.8, n_samples)

# Create a DataFrame with all these variables
df_numerical = pd.DataFrame({
    'Height (cm)': heights,
    'Children': children,
    'Temperature (°C)': temperatures,
    'Income': income
})

# Display summary statistics
print("Dataset summary statistics:")
df_numerical.describe()

## 3. Histogram Visualization

Histograms are one of the most common ways to visualize the distribution of a numerical variable. They divide the data into bins and show the frequency of observations in each bin.

Key considerations for histograms:
- Bin size/count: Affects the smoothness and detail of the visualization
- Normalization: Raw counts vs. density/probability
- Overlaid elements: Like kernel density estimate (KDE) curves

In [ ]:
# Create histograms for our continuous variable (height) with different bin sizes
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Effect of Bin Size on Histogram Visualization', fontsize=16)

# Few bins
sns.histplot(df_numerical['Height (cm)'], bins=5, kde=True, ax=axes[0, 0])
axes[0, 0].set_title('5 bins (too few)')

# Medium bins
sns.histplot(df_numerical['Height (cm)'], bins=15, kde=True, ax=axes[0, 1])
axes[0, 1].set_title('15 bins (balanced)')

# Many bins
sns.histplot(df_numerical['Height (cm)'], bins=50, kde=True, ax=axes[1, 0])
axes[1, 0].set_title('50 bins (too many)')

# Automatic bin selection (Freedman-Diaconis rule)
sns.histplot(df_numerical['Height (cm)'], kde=True, ax=axes[1, 1])
axes[1, 1].set_title('Auto bins (default)')

plt.tight_layout()
plt.subplots_adjust(top=0.9)

In [ ]:
# Compare histograms for different numerical data types
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Histograms for Different Numerical Data Types', fontsize=16)

# Continuous data (height)
sns.histplot(df_numerical['Height (cm)'], kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Continuous Data (Height)')

# Discrete data (children)
sns.histplot(df_numerical['Children'], discrete=True, kde=False, ax=axes[0, 1])
axes[0, 1].set_title('Discrete Data (Children)')

# Interval data (temperature)
sns.histplot(df_numerical['Temperature (°C)'], kde=True, ax=axes[1, 0])
axes[1, 0].set_title('Interval Data (Temperature)')

# Ratio data with log scale (income)
sns.histplot(df_numerical['Income'], kde=True, log_scale=True, ax=axes[1, 1])
axes[1, 1].set_title('Ratio Data with Log Scale (Income)')

plt.tight_layout()
plt.subplots_adjust(top=0.9)

In [ ]:
# Create an interactive histogram with Plotly
fig = px.histogram(df_numerical, x='Height (cm)', nbins=20, 
                   marginal='box',  # adds a box plot on the margin
                   opacity=0.7,
                   histnorm='probability density',  # normalize to probability density
                   title='Interactive Histogram with Plotly')

# Add KDE curve
fig.add_trace(go.Scatter(
    x=np.linspace(df_numerical['Height (cm)'].min(), df_numerical['Height (cm)'].max(), 100),
    y=sns.kdeplot(df_numerical['Height (cm)']).get_lines()[0].get_ydata(),
    mode='lines',
    name='KDE',
    line=dict(color='red', width=2)
))

fig.update_layout(
    xaxis_title="Height (cm)",
    yaxis_title="Probability Density",
    legend_title="Distribution",
    width=800,
    height=500
)

fig.show()

## 4. Box Plots for Distribution Analysis

Box plots (also known as box-and-whisker plots) are excellent for visualizing:
- The central tendency (median)
- Dispersion (interquartile range)
- Skewness (position of median within the box)
- Outliers (points beyond the whiskers)

They're particularly useful for comparing distributions between different groups.

In [ ]:
# Load the tips dataset from seaborn for comparison
tips = load_dataset('tips')

# Basic box plot
plt.figure(figsize=(10, 6))
sns.boxplot(data=tips, x='day', y='total_bill')
plt.title('Box Plot: Total Bill by Day')
plt.ylabel('Total Bill ($)')
plt.xlabel('Day')
plt.show()

In [ ]:
# Compare box plots with notches (for statistical inference) and add data points
plt.figure(figsize=(12, 6))
sns.boxplot(data=tips, x='day', y='total_bill', hue='sex', notch=True, palette='Set2')
# Add individual data points with jitter for better visibility
sns.stripplot(data=tips, x='day', y='total_bill', hue='sex', dodge=True, 
              alpha=0.5, jitter=0.3, size=4, palette='Set2', legend=False)
plt.title('Box Plot with Notches and Individual Data Points: Total Bill by Day and Sex')
plt.ylabel('Total Bill ($)')
plt.xlabel('Day')
plt.legend(title='Sex', loc='upper right')
plt.show()

In [ ]:
# Create a horizontal box plot comparing multiple variables from our numerical dataset
plt.figure(figsize=(10, 8))
# Melt the DataFrame to get all numerical variables in one column
melted_df = df_numerical.copy()
# Standardize values to be able to compare on same scale
for col in melted_df.columns:
    melted_df[col] = (melted_df[col] - melted_df[col].mean()) / melted_df[col].std()
    
melted_df = pd.melt(melted_df, var_name='Variable', value_name='Standardized Value')

# Create the box plot
sns.boxplot(data=melted_df, x='Standardized Value', y='Variable', orient='h', palette='viridis')
plt.title('Comparing Standardized Distributions Across Variables')
plt.tight_layout()
plt.show()

In [ ]:
# Interactive box plot with Plotly
fig = px.box(tips, x='day', y='total_bill', color='time',
             title='Interactive Box Plot: Total Bill by Day and Time',
             points='all',  # show all points
             notched=True)  # add notches for confidence intervals

fig.update_layout(
    xaxis_title='Day',
    yaxis_title='Total Bill ($)',
    legend_title='Time',
    boxmode='group',
    width=800, 
    height=500
)

fig.show()

## 5. Violin Plots for Density Visualization

Violin plots combine box plot features with kernel density estimation to show the full distribution shape. They are particularly useful for:
- Visualizing the distribution's shape (unimodal, bimodal, etc.)
- Comparing distributions across different groups
- Identifying areas of high density in the data

In [ ]:
# Basic violin plot
plt.figure(figsize=(10, 6))
sns.violinplot(data=tips, x='day', y='total_bill')
plt.title('Violin Plot: Total Bill by Day')
plt.ylabel('Total Bill ($)')
plt.xlabel('Day')
plt.show()

In [ ]:
# Compare violin plots with box plots side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Comparing Box Plots and Violin Plots', fontsize=16)

# Box plot
sns.boxplot(data=tips, x='day', y='total_bill', ax=axes[0])
axes[0].set_title('Box Plot')
axes[0].set_ylabel('Total Bill ($)')
axes[0].set_xlabel('Day')

# Violin plot
sns.violinplot(data=tips, x='day', y='total_bill', ax=axes[1])
axes[1].set_title('Violin Plot')
axes[1].set_ylabel('Total Bill ($)')
axes[1].set_xlabel('Day')

plt.tight_layout()
plt.subplots_adjust(top=0.85)

In [ ]:
# Split violin plot by a categorical variable
plt.figure(figsize=(12, 6))
sns.violinplot(data=tips, x='day', y='total_bill', hue='sex', split=True, inner='quart', palette='Set2')
plt.title('Split Violin Plot: Total Bill by Day and Sex')
plt.ylabel('Total Bill ($)')
plt.xlabel('Day')
plt.legend(title='Sex', loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# Advanced violin plot with inner points and box plot
plt.figure(figsize=(12, 6))
sns.violinplot(data=tips, x='day', y='total_bill', hue='time', inner='box', palette='pastel')
# Add individual points with jitter
sns.stripplot(data=tips, x='day', y='total_bill', hue='time', dodge=True, 
              alpha=0.5, jitter=0.3, size=3, palette='dark', legend=False)
plt.title('Violin Plot with Inner Box Plot and Individual Points')
plt.ylabel('Total Bill ($)')
plt.xlabel('Day')
plt.legend(title='Time', loc='upper right')
plt.tight_layout()
plt.show()

## 6. KDE Plots (Kernel Density Estimation)

Kernel Density Estimation (KDE) plots visualize the probability density function of a variable. They're useful for:
- Smoothing histograms to create continuous distribution curves
- Identifying modes (peaks) in the data
- Comparing the shapes of different distributions

In [ ]:
# Basic KDE plot
plt.figure(figsize=(10, 6))
sns.kdeplot(data=tips, x='total_bill', fill=True)
plt.title('KDE Plot: Distribution of Total Bill')
plt.xlabel('Total Bill ($)')
plt.ylabel('Density')
plt.tight_layout()
plt.show()

In [ ]:
# Compare KDE plots for different groups
plt.figure(figsize=(10, 6))
sns.kdeplot(data=tips, x='total_bill', hue='time', fill=True, common_norm=False, palette='crest', alpha=.5)
plt.title('KDE Plot: Distribution of Total Bill by Time')
plt.xlabel('Total Bill ($)')
plt.ylabel('Density')
plt.tight_layout()
plt.show()

In [ ]:
# Bivariate KDE plot
plt.figure(figsize=(10, 8))
sns.kdeplot(data=tips, x='total_bill', y='tip', fill=True, cmap='viridis', thresh=0.05)
plt.title('Bivariate KDE Plot: Total Bill vs. Tip Amount')
plt.xlabel('Total Bill ($)')
plt.ylabel('Tip Amount ($)')
plt.tight_layout()
plt.show()

In [ ]:
# KDE plots with different bandwidths
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Effect of Bandwidth on KDE Plots', fontsize=16)

# Small bandwidth
sns.kdeplot(data=tips, x='total_bill', ax=axes[0, 0], bw_adjust=0.2, fill=True)
axes[0, 0].set_title('Small Bandwidth (bw_adjust=0.2)')

# Default bandwidth
sns.kdeplot(data=tips, x='total_bill', ax=axes[0, 1], bw_adjust=1.0, fill=True)
axes[0, 1].set_title('Default Bandwidth (bw_adjust=1.0)')

# Large bandwidth
sns.kdeplot(data=tips, x='total_bill', ax=axes[1, 0], bw_adjust=2.0, fill=True)
axes[1, 0].set_title('Large Bandwidth (bw_adjust=2.0)')

# Very large bandwidth (oversmoothed)
sns.kdeplot(data=tips, x='total_bill', ax=axes[1, 1], bw_adjust=5.0, fill=True)
axes[1, 1].set_title('Very Large Bandwidth (bw_adjust=5.0)')

plt.tight_layout()
plt.subplots_adjust(top=0.9)

## 7. Scatter Plots for Relationships

Scatter plots are fundamental for visualizing relationships between two numerical variables. They help identify:
- Correlation and relationship patterns
- Clusters and groupings
- Outliers and anomalies
- Trends over a continuous variable

In [ ]:
# Load the iris dataset for multi-feature visualization
iris = load_dataset('iris')

# Basic scatter plot
plt.figure(figsize=(10, 6))
sns.scatterplot(data=iris, x='sepal_length', y='sepal_width', hue='species', palette='viridis')
plt.title('Scatter Plot: Sepal Length vs Width by Species')
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Sepal Width (cm)')
plt.legend(title='Species')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot with regression line
plt.figure(figsize=(10, 6))
sns.regplot(data=tips, x='total_bill', y='tip', scatter_kws={'alpha':0.5}, line_kws={'color':'red'})
plt.title('Scatter Plot with Regression Line: Total Bill vs Tip')
plt.xlabel('Total Bill ($)')
plt.ylabel('Tip Amount ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot with multiple dimensions (size and color)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=tips, x='total_bill', y='tip', hue='day', size='size', sizes=(20, 200), palette='deep')
plt.title('Multi-dimensional Scatter Plot: Total Bill vs Tip')
plt.xlabel('Total Bill ($)')
plt.ylabel('Tip Amount ($)')
plt.legend(title='Day', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Interactive scatter plot with Plotly
fig = px.scatter(iris, x='sepal_length', y='sepal_width', 
                 color='species', size='petal_width', 
                 hover_data=['petal_length'],
                 title='Interactive Multi-dimensional Scatter Plot',
                 labels={'sepal_length': 'Sepal Length (cm)', 
                         'sepal_width': 'Sepal Width (cm)',
                         'species': 'Species',
                         'petal_width': 'Petal Width (cm)',
                         'petal_length': 'Petal Length (cm)'},
                 color_discrete_sequence=px.colors.qualitative.Vivid)

fig.update_layout(
    legend_title='Species',
    width=800,
    height=600
)

fig.show()

## 8. Heatmaps for Correlation Analysis

Heatmaps are excellent for visualizing the correlation between multiple numerical variables in a dataset. They use color intensity to represent correlation strength, making it easy to identify patterns and relationships.

In [ ]:
# Create correlation matrix for the iris dataset
correlation_matrix = iris.drop('species', axis=1).corr()

# Create a basic heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix Heatmap: Iris Features')
plt.tight_layout()
plt.show()

In [ ]:
# Create correlation matrix for tips dataset
corr_tips = tips.select_dtypes(include=[np.number]).corr()

# Customized heatmap with mask for upper triangle
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_tips, dtype=bool))
sns.heatmap(corr_tips, annot=True, mask=mask, cmap='YlGnBu', 
            linewidths=0.5, annot_kws={'size': 10}, fmt='.2f')
plt.title('Half-masked Correlation Matrix Heatmap: Tips Dataset')
plt.tight_layout()
plt.show()

In [ ]:
# Generate a larger numerical dataset
np.random.seed(42)
n_features = 15
n_samples = 200

# Create correlated features
corr_data = np.random.randn(n_samples, n_features)
# Introduce some correlations
corr_data[:, 1] = corr_data[:, 0] * 0.8 + np.random.randn(n_samples) * 0.2
corr_data[:, 3] = corr_data[:, 2] * -0.7 + np.random.randn(n_samples) * 0.3
corr_data[:, 6] = corr_data[:, 5] * 0.6 + corr_data[:, 4] * 0.3 + np.random.randn(n_samples) * 0.1
corr_df = pd.DataFrame(corr_data, columns=[f'Feature_{i}' for i in range(n_features)])

# Create correlation matrix
corr_matrix = corr_df.corr()

# Clustered heatmap with dendrogram
plt.figure(figsize=(14, 12))
sns.clustermap(corr_matrix, 
               cmap='RdBu_r', 
               annot=True, 
               fmt='.2f',
               linewidths=0.5,
               figsize=(14, 12),
               vmin=-1, vmax=1,
               annot_kws={'size': 8})
plt.title('Clustered Correlation Heatmap with Dendrogram', pad=50)
plt.tight_layout()
plt.show()

In [ ]:
# Interactive heatmap with Plotly
fig = px.imshow(corr_matrix,
                labels=dict(x="Feature", y="Feature", color="Correlation"),
                x=[f'Feature_{i}' for i in range(n_features)],
                y=[f'Feature_{i}' for i in range(n_features)],
                color_continuous_scale='RdBu_r',
                title='Interactive Correlation Heatmap')

fig.update_layout(
    width=750,
    height=700,
    coloraxis_colorbar=dict(
        title="Correlation",
        thicknessmode="pixels", thickness=20,
        lenmode="pixels", len=300,
        tickvals=[-1, -0.5, 0, 0.5, 1],
        ticktext=['-1', '-0.5', '0', '0.5', '1']
    )
)

# Add correlation values as text
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        fig.add_annotation(
            x=i, y=j,
            text=str(round(corr_matrix.iloc[j, i], 2)),
            showarrow=False,
            font=dict(color="black" if abs(corr_matrix.iloc[j, i]) < 0.7 else "white", size=9)
        )

fig.show()

## 9. Pair Plots for Multiple Variables

Pair plots (also known as scatter plot matrices) provide a comprehensive way to visualize relationships between multiple numerical variables. They're excellent for initial exploratory data analysis to identify patterns and relationships across multiple dimensions.

In [ ]:
# Basic pair plot
sns.pairplot(iris, hue='species', palette='viridis', height=2.5)
plt.suptitle('Pair Plot of Iris Dataset', y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Customized pair plot with different diagonal and non-diagonal plots
sns.pairplot(iris, 
             hue='species', 
             diag_kind='kde',  # KDE plots on diagonal
             plot_kws={'alpha': 0.6, 's': 30, 'edgecolor': 'k', 'linewidth': 0.5},
             diag_kws={'fill': True, 'alpha': 0.5, 'linewidth': 1},
             palette='tab10',
             corner=True)  # Only show lower triangle
plt.suptitle('Customized Pair Plot with KDE Diagonals', y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Create custom pair grid with different plots
g = sns.PairGrid(tips, hue='time', palette='Set2', height=2.5)
# Scatter plots in the lower triangle
g.map_lower(sns.scatterplot, alpha=0.7, s=30, edgecolor='white', linewidth=0.5)
# Kernel density plots in the diagonal
g.map_diag(sns.kdeplot, fill=True, alpha=0.7)
# Regression plots in the upper triangle
g.map_upper(sns.regplot, scatter_kws={'alpha':0.3, 's':10}, line_kws={'color':'red'})
g.add_legend()
g.fig.suptitle('Custom Pair Grid: Tips Dataset', y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Interactive pair plot with Plotly
fig = px.scatter_matrix(
    iris, 
    dimensions=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'],
    color='species',
    symbol='species',
    labels={col: col.replace('_', ' ').title() for col in iris.columns},
    title="Interactive Scatter Matrix (Pair Plot): Iris Dataset",
    opacity=0.7
)

# Update axis labels
for axis in fig.layout:
    if type(fig.layout[axis]) == go.layout.XAxis:
        fig.layout[axis].title.font.size = 12
    if type(fig.layout[axis]) == go.layout.YAxis:
        fig.layout[axis].title.font.size = 12

fig.update_traces(diagonal_visible=False)
fig.update_layout(
    width=950,
    height=950
)

fig.show()

## 10. 3D Visualizations for Numerical Data

3D visualizations can help reveal complex relationships in numerical data that might not be apparent in 2D plots, especially when dealing with multiple variables.

In [ ]:
# 3D scatter plot with matplotlib
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Create a colormap based on the species
colors = {'setosa': 'r', 'versicolor': 'g', 'virginica': 'b'}
markers = {'setosa': 'o', 'versicolor': '^', 'virginica': 's'}

for species, group in iris.groupby('species'):
    ax.scatter(group['sepal_length'], group['sepal_width'], group['petal_length'],
               label=species, color=colors[species], marker=markers[species], s=50, alpha=0.7)

ax.set_xlabel('Sepal Length (cm)')
ax.set_ylabel('Sepal Width (cm)')
ax.set_zlabel('Petal Length (cm)')
ax.set_title('3D Scatter Plot of Iris Dataset', fontsize=14)
ax.legend()

# Improve perspective
ax.view_init(elev=30, azim=45)
plt.tight_layout()
plt.show()

In [ ]:
# Create a 3D surface plot using a function
from matplotlib import cm

# Create data for the 3D plot
x = np.linspace(-5, 5, 100)
y = np.linspace(-5, 5, 100)
x, y = np.meshgrid(x, y)
z = np.sin(np.sqrt(x**2 + y**2))

# Create the plot
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot the surface
surf = ax.plot_surface(x, y, z, cmap=cm.viridis, linewidth=0, antialiased=True, alpha=0.8)

# Add a color bar
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)

# Set labels and title
ax.set_xlabel('X axis')
ax.set_ylabel('Y axis')
ax.set_zlabel('Z axis')
ax.set_title('3D Surface Plot Example', fontsize=14)

# Improve perspective
ax.view_init(elev=30, azim=45)
plt.tight_layout()
plt.show()

In [ ]:
# Interactive 3D scatter plot with Plotly
fig = px.scatter_3d(
    iris, 
    x='sepal_length', 
    y='sepal_width', 
    z='petal_length',
    color='species',
    symbol='species',
    opacity=0.7,
    size='petal_width',
    size_max=10,
    labels={
        'sepal_length': 'Sepal Length (cm)',
        'sepal_width': 'Sepal Width (cm)',
        'petal_length': 'Petal Length (cm)',
        'petal_width': 'Petal Width (cm)',
        'species': 'Species'
    },
    title='Interactive 3D Scatter Plot: Iris Dataset'
)

fig.update_layout(
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ),
    width=900,
    height=700
)

fig.show()

In [ ]:
# Interactive 3D Surface Plot with Plotly
fig = go.Figure(data=[go.Surface(z=z, x=x, y=y, colorscale='viridis')])

fig.update_layout(
    title='Interactive 3D Surface Plot',
    autosize=False,
    width=900,
    height=700,
    scene=dict(
        xaxis_title='X Axis',
        yaxis_title='Y Axis',
        zaxis_title='Z Axis',
        aspectratio=dict(x=1, y=1, z=0.7),
        camera=dict(
            eye=dict(x=1.8, y=1.8, z=1.2)
        )
    )
)

fig.show()

## Summary

In this notebook, we've explored various techniques for visualizing numerical data:

1. **Histograms**: Excellent for understanding the distribution and frequency of numerical data
2. **Box Plots**: Great for summarizing distributions and identifying outliers
3. **Violin Plots**: Combine features of box plots and KDE plots to show the full distribution shape
4. **KDE Plots**: Visualize the probability density function of numerical variables
5. **Scatter Plots**: Essential for exploring relationships between numerical variables
6. **Heatmaps**: Perfect for visualizing correlations between multiple variables
7. **Pair Plots**: Provide a comprehensive view of relationships across multiple dimensions
8. **3D Visualizations**: Help reveal complex relationships in multidimensional data

Key considerations when visualizing numerical data:
- Choose the appropriate visualization based on the data type and analytical goals
- Pay attention to scales (linear vs. logarithmic) depending on data distribution
- Use color, size, and shape effectively to represent additional dimensions
- Consider interactive visualizations for complex data exploration
- Always ensure your visualizations are clear, accurate, and accessible